# Customer Support — Resolution Time Regression

## Step 1: Problem Definition
Predict **resolution time in hours** to support SLA planning and staffing.

In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

sns.set_theme(style='whitegrid')
print('Project root:', PROJECT_ROOT)

## Step 2: Data Collection

In [ ]:
from src.data_loader import load_tickets
raw_df = load_tickets(use_db=True)
raw_df.head()

## Step 3: Data Cleaning
Drop leakage columns (resolution notes, status, resolved date, satisfaction).

In [ ]:
from src.preprocessor import clean_data
cleaned_df = clean_data(raw_df, task_type='regression')
cleaned_df['resolution_time_hours'].describe()

### EDA: Resolution time distribution

In [ ]:
import seaborn as sns
sns.histplot(cleaned_df['resolution_time_hours'], kde=True, bins=40)
plt.title('Resolution Time Distribution')
plt.show()

## Step 4: Feature Engineering
Cap outliers, derive date features, keep text for TF-IDF.

In [ ]:
from src.preprocessor import engineer_features
from src.label_engineering import derive_resolution_hours
import numpy as np

fe_df = engineer_features(cleaned_df, task_type='regression')
fe_df = fe_df.sample(min(30000, len(fe_df)), random_state=42)
fe_df['resolution_time_hours'] = derive_resolution_hours(fe_df)
print('Rows:', len(fe_df))

## Step 5: Train-Test Split
Log-transform target for stable regression.

In [ ]:
from sklearn.model_selection import train_test_split

X = fe_df.drop(columns=['resolution_time_hours'])
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].fillna('Unknown')
y = np.log1p(fe_df['resolution_time_hours'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape)

## Step 6: Model Selection
Linear Regression, Random Forest, and MLP neural network.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
from src.model_trainer import get_regression_pipelines

for name, pipe in get_regression_pipelines(X_train).items():
    pipe.fit(X_train, y_train)
    pred = np.expm1(pipe.predict(X_test))
    true = np.expm1(y_test)
    print(f"{name}: R2={r2_score(true, pred):.4f}  RMSE={mean_squared_error(true, pred, squared=False):.2f}")

## Step 7: Model Training

In [ ]:
reg = get_regression_pipelines(X_train)['Random Forest']
reg.fit(X_train, y_train)

## Step 8: Model Evaluation

In [ ]:
y_pred = np.expm1(reg.predict(X_test))
y_true = np.expm1(y_test)
print('R2:', r2_score(y_true, y_pred))
plt.scatter(y_true, y_pred, alpha=0.2)
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title('Actual vs Predicted Resolution Time')
plt.show()

## Step 9: Model Tuning
GridSearchCV on Random Forest (optional).

In [ ]:
print('Define param_grid for regressor tuning as needed.')

## Step 10: Deployment

In [ ]:
import joblib
from src.paths import get_models_dir

r2 = r2_score(y_true, y_pred)
bundle = {'model': reg, 'model_name': 'Random Forest', 'r2': r2, 'log_target': True}
joblib.dump(bundle, os.path.join(get_models_dir(), 'regression_model.pkl'))
print('Saved regression model. R2=', r2)